# Aula 05 - Sequenciamento Genético

## Sequenciamento COVID

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession \
        .builder \
        .master("local[*]") \
        .appName("DNA_Michel2025_segunda") \
        .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/03 23:12:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/03 23:12:30 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/11/03 23:12:30 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/11/03 23:12:30 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
25/11/03 23:12:30 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
25/11/03 23:12:30 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.
25/11/03 23:12:30 WARN Utils: Service 'SparkUI' could not bind on port 4045. Attempting port 4046.
25/11/03 23:12:30 WARN Utils: Service 'SparkUI' could not bind on port 4046. Attempting port 4047.
25/11/03 23:12:30 WARN Utils: Serv

In [3]:
spark

In [4]:
sc = spark.sparkContext

### Vírus de Wuhan

SARS-CoV-2-Wuhan-NC_045512.2.fasta

In [5]:
path = "../../dados/10_dados/covid_dna/"

In [7]:
!ls ../../dados/10_dados/covid_dna/

SARS-CoV-2-Washington_MT293201.1.fasta	SARS-CoV-2-Wuhan-NC_045512.2.fasta


In [8]:
covidRDD = sc.textFile(path + "SARS-CoV-2-Wuhan-NC_045512.2.fasta")

In [10]:
covidRDD.take(10)

['>NC_045512.2 Severe acute respiratory syndrome coronavirus 2 isolate Wuhan-Hu-1, complete genome',
 'ATTAAAGGTTTATACCTTCCCAGGTAACAAACCAACCAACTTTCGATCTCTTGTAGATCTGTTCTCTAAA',
 'CGAACTTTAAAATCTGTGTGGCTGTCACTCGGCTGCATGCTTAGTGCACTCACGCAGTATAATTAATAAC',
 'TAATTACTGTCGTTGACAGGACACGAGTAACTCGTCTATCTTCTGCAGGCTGCTTACGGTTTCGTCCGTG',
 'TTGCAGCCGATCATCAGCACATCTAGGTTTCGTCCGGGTGTGACCGAAAGGTAAGATGGAGAGCCTTGTC',
 'CCTGGTTTCAACGAGAAAACACACGTCCAACTCAGTTTGCCTGTTTTACAGGTTCGCGACGTGCTCGTAC',
 'GTGGCTTTGGAGACTCCGTGGAGGAGGTCTTATCAGAGGCACGTCAACATCTTAAAGATGGCACTTGTGG',
 'CTTAGTAGAAGTTGAAAAAGGCGTTTTGCCTCAACTTGAACAGCCCTATGTGTTCATCAAACGTTCGGAT',
 'GCTCGAACTGCACCTCATGGTCATGTTATGGTTGAGCTGGTAGCAGAACTCGAAGGCATTCAGTACGGTC',
 'GTAGTGGTGAGACACTTGGTGTCCTTGTCCCTCATGTGGGCGAAATACCAGTGGCTTACCGCAAGGTTCT']

In [11]:
covidRDD.count()

430

In [12]:
covidRDD = covidRDD.filter(lambda x: not x.startswith(">"))

In [13]:
covidRDD.take(10)

['ATTAAAGGTTTATACCTTCCCAGGTAACAAACCAACCAACTTTCGATCTCTTGTAGATCTGTTCTCTAAA',
 'CGAACTTTAAAATCTGTGTGGCTGTCACTCGGCTGCATGCTTAGTGCACTCACGCAGTATAATTAATAAC',
 'TAATTACTGTCGTTGACAGGACACGAGTAACTCGTCTATCTTCTGCAGGCTGCTTACGGTTTCGTCCGTG',
 'TTGCAGCCGATCATCAGCACATCTAGGTTTCGTCCGGGTGTGACCGAAAGGTAAGATGGAGAGCCTTGTC',
 'CCTGGTTTCAACGAGAAAACACACGTCCAACTCAGTTTGCCTGTTTTACAGGTTCGCGACGTGCTCGTAC',
 'GTGGCTTTGGAGACTCCGTGGAGGAGGTCTTATCAGAGGCACGTCAACATCTTAAAGATGGCACTTGTGG',
 'CTTAGTAGAAGTTGAAAAAGGCGTTTTGCCTCAACTTGAACAGCCCTATGTGTTCATCAAACGTTCGGAT',
 'GCTCGAACTGCACCTCATGGTCATGTTATGGTTGAGCTGGTAGCAGAACTCGAAGGCATTCAGTACGGTC',
 'GTAGTGGTGAGACACTTGGTGTCCTTGTCCCTCATGTGGGCGAAATACCAGTGGCTTACCGCAAGGTTCT',
 'TCTTCGTAAGAACGGTAATAAAGGAGCTGGTGGCCATAGTTACGGCGCCGATCTAAAGTCATTTGACTTA']

In [14]:
def process_fasta(record):
    key_value_list = []
    chars = record.lower()
    for c in chars:
        key_value_list.append( (c, 1) )
    return key_value_list

In [15]:
covidRDD1 = covidRDD.flatMap(lambda x: process_fasta(x))

In [16]:
covidRDD1.take(10)

[('a', 1),
 ('t', 1),
 ('t', 1),
 ('a', 1),
 ('a', 1),
 ('a', 1),
 ('g', 1),
 ('g', 1),
 ('t', 1),
 ('t', 1)]

In [17]:
covidRDD2 = covidRDD1.reduceByKey(lambda a, b: a+b)

In [20]:
covidRDD2.take(10)

[('t', 9594), ('g', 5863), ('c', 5492), ('a', 8954)]

In [21]:
total = covidRDD2.map(lambda x: x[1]).sum()
total

29903

In [22]:
freq = covidRDD2.map(lambda x: (x[0], x[1]/total*100))

In [23]:
%%time
freq.collect()

CPU times: user 2.94 ms, sys: 2.01 ms, total: 4.95 ms
Wall time: 134 ms


[('t', 32.083737417650404),
 ('g', 19.60672842189747),
 ('c', 18.366050229074006),
 ('a', 29.943483931378122)]

### Vírus de Washington

SARS-CoV-2-Washington_MT293201.1.fasta

In [24]:
covidRDD = sc.textFile(path + "SARS-CoV-2-Washington_MT293201.1.fasta")
covidRDD = covidRDD.filter(lambda x: not x.startswith(">"))
covidRDD1 = covidRDD.flatMap(lambda x: process_fasta(x) )
covidRDD2 = covidRDD1.reduceByKey(lambda x, y: x+y)
total = covidRDD2.map(lambda x: x[1]).sum()
freq = covidRDD2.map(lambda x: (x[0], x[1]/total*100))
freq.collect()

[('c', 18.360919386182402),
 ('t', 32.14166052402332),
 ('g', 19.63077129263553),
 ('a', 29.866648797158746)]

## Estratégias de sequenciamento

### Estratégia 01 (igual anteriormente)

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession \
        .builder \
        .master("local[*]") \
        .appName("DNA_Michel_Otimizacoes2") \
        .getOrCreate()

In [ ]:
spark

In [ ]:
sc = spark.sparkContext

### Estratégia 2 - Map/Reduce Otimizado

Alterando a estruturada de dados (de lista para dicionário)

### Estratégia 3 - Usando mapPartitions